# Rithmic Contracts And 1-Minute Bars To Nautilus Parquet

## User guide

This notebook supports two history workflows for a single Rithmic futures root, defaulting to `MNQ.CME`:

1. Discover available specific contracts by probing candidate Rithmic contract symbols.
2. Download native 1-minute historical bars either:
   - per specific contract, or
   - as a continuous root request using the root symbol itself, such as `MNQ`.
3. Run a simple EMA-crossover backtest against the synthetic continuous root bars to sanity-check the downloaded history.

## What gets written

- `INSTRUMENTS_PARQUET_PATH`: one parquet file containing Nautilus `FuturesContract` objects for the discovered real contracts plus one extra synthetic root definition such as `MNQ.CME`. That extra definition is an imaginary backtest-only continuous instrument, not a venue-native contract.
- `BARS_OUTPUT_DIR/YYYY-MM-DD.parquet`: one parquet file per UTC day containing contract-specific Nautilus `Bar` objects.
- `CONTINUOUS_BARS_OUTPUT_DIR/YYYY-MM-DD.parquet`: one parquet file per UTC day containing continuous root Nautilus `Bar` objects.

## Credentials and environment

- Run the first code cell every time. It imports the notebook dependencies and defines the helper functions used later.
- `.env` loading is optional. If a repo-root `.env` file exists, the notebook loads it.
- If no `.env` file exists, exported environment variables still work. The notebook ultimately reads credentials through `RithmicDataClientConfig.from_env(...)`.
- `PROFILE` is optional and only matters if you use `RITHMIC_{PROFILE}_*` variables.

## Recommended run order

1. Run the bootstrap cell.
2. Edit the settings cell.
3. Run the contract-request preview cell and verify the symbol candidates look sane.
4. Run the discovery cell and inspect `discovery_summary_df`.
5. Run the specific-contract bar cell if you want bars keyed to each real contract.
6. Run the optional continuous-root bar cell if you also want root-symbol history keyed to the synthetic root instrument.
7. Run the final EMA-crossover backtest cell if you want a quick validation pass on the continuous root data only.

## Important behavior

- Contract discovery uses explicit symbol probing, not the exchange-wide snapshot path.
- The discovery cell writes one extra synthetic root instrument by copying the first successful real contract definition, renaming it to the root symbol, and setting its expiration to the maximum pandas timestamp so it does not expire during long backtests.
- Historical bar requests resume from the last returned bar timestamp because Rithmic responses can be truncated.
- During bar downloads, each instrument/day batch is written to parquet immediately after the download completes so memory usage stays bounded.
- Storage is Nautilus-native via `ArrowSerializer`; pandas is only used for notebook previews.
- This synthetic root is only for historical backtests where root-symbol history is deeper than specific-contract history. In this workflow the continuous root series is expected to extend back to 2019-06.
- Rithmic history usage is plan-limited, so keep the date range bounded.


## Step 1: Bootstrap the notebook

What this cell does:
- Imports Nautilus, Rithmic, pandas, and parquet dependencies.
- Finds the repo root.
- Optionally loads `.env` from the repo root.
- Defines `REPO_ROOT` for later path configuration.

What to expect:
- A message saying whether `.env` was loaded or skipped.
- The final line displays `REPO_ROOT`.
- If imports fail here, fix the environment before running anything else.


In [7]:
from __future__ import annotations

import os
from collections import defaultdict
from datetime import UTC, datetime, timedelta
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from nautilus_trader.adapters.rithmic import RithmicDataClientConfig
from nautilus_trader.adapters.rithmic.bindings import RithmicDataClient
from nautilus_trader.adapters.rithmic.bindings import RithmicGateway
from nautilus_trader.adapters.rithmic.bindings import (
    RithmicInstrumentProvider as RustRithmicInstrumentProvider,
)
from nautilus_trader.adapters.rithmic.config import to_binding_environment
from nautilus_trader.adapters.rithmic.data import RithmicLiveDataClient
from nautilus_trader.adapters.rithmic.providers import (
    RithmicInstrumentProvider as PythonRithmicInstrumentProvider,
)
from nautilus_trader.model.data import Bar
from nautilus_trader.model.data import BarType
from nautilus_trader.model.instruments import FuturesContract
from nautilus_trader.serialization.arrow.serializer import ArrowSerializer


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Could not find repo root. Launch Jupyter from inside the repo.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

ENV_PATH = REPO_ROOT / '.env'


def load_env_file() -> bool:
    if load_dotenv is not None and ENV_PATH.exists():
        load_dotenv(ENV_PATH)
        print(f'Loaded environment from {ENV_PATH}')
        return True
    if load_dotenv is None and ENV_PATH.exists():
        for raw in ENV_PATH.read_text().splitlines():
            raw = raw.strip()
            if not raw or raw.startswith('#') or '=' not in raw:
                continue
            key, _, value = raw.partition('=')
            os.environ.setdefault(key.strip(), value.strip().strip("'").strip('"'))
        print(f'Loaded environment from {ENV_PATH} (fallback parser)')
        return True
    print(f'No .env found at {ENV_PATH}, skipping')
    return False


load_env_file()

REPO_ROOT


Loaded environment from /Users/agedvagabond/Developer/nautilus_trader_env/.env


PosixPath('/Users/agedvagabond/Developer/nautilus_trader_env')

## Step 2: Configure the notebook

What this cell does:
- Centralizes all user-editable settings for the notebook.
- Lets you change the root product, exchange, output directory, discovery window, and bar-download window.
- Defines separate output locations for contract-specific bars and optional continuous-root bars.
- Defines the EMA-crossover backtest settings used by the final validation cell.
- Keeps `TRADEABLE_ONLY=False` by default.

What to expect:
- A printed summary of the chosen settings and output paths.
- Only edit this cell when changing notebook behavior; later cells should not need manual edits.


In [8]:
from datetime import date

PROFILE = os.environ.get('RITHMIC_PROFILE') or None

ROOT_PRODUCT = 'MNQ'
ROOT_EXCHANGE = 'CME'
OUTPUT_DIR = (REPO_ROOT / 'data' / 'rithmic').resolve()

DISCOVERY_START_YEAR = 2019
DISCOVERY_END_YEAR = date.today().year + 1
CONTRACT_MONTH_CODES = {
    'F': 1,
    'G': 2,
    'H': 3,
    'J': 4,
    'K': 5,
    'M': 6,
    'N': 7,
    'Q': 8,
    'U': 9,
    'V': 10,
    'X': 11,
    'Z': 12,
}
TRADEABLE_ONLY = False
MERGE_WITH_EXISTING_INSTRUMENTS = True

BAR_INSTRUMENT_LIMIT: int | None = None
BAR_DAY_LIMIT: int | None = None
BAR_MAX_REQUEST_PAGES = 64
BAR_START = pd.Timestamp('2024-01-01 00:00:00', tz='UTC')
BAR_END = pd.Timestamp.now(tz='UTC').floor('min')
MERGE_WITH_EXISTING_BARS = True
SKIP_EXISTING_BAR_FILES = False

BACKTEST_START = BAR_START
BACKTEST_END = BAR_END
BACKTEST_FAST_EMA_PERIOD = 10
BACKTEST_SLOW_EMA_PERIOD = 20
BACKTEST_TRADE_SIZE = '1'
BACKTEST_STARTING_BALANCE = 250_000.0
BACKTEST_REPORT_PREVIEW_ROWS = 20

ROOT_PRODUCT_SLUG = ROOT_PRODUCT.lower()
INSTRUMENTS_PARQUET_PATH = OUTPUT_DIR / f'{ROOT_PRODUCT_SLUG}_futures_contracts.parquet'
BARS_OUTPUT_DIR = OUTPUT_DIR / f'{ROOT_PRODUCT_SLUG}_bars_1minute_by_day'
CONTINUOUS_BARS_OUTPUT_DIR = OUTPUT_DIR / f'{ROOT_PRODUCT_SLUG}_continuous_bars_1minute_by_day'

print(f'Profile: {PROFILE or "<default RITHMIC_*>"}')
print(f'Root product: {ROOT_PRODUCT}')
print(f'Root exchange: {ROOT_EXCHANGE}')
print(f'Discovery years: {DISCOVERY_START_YEAR} -> {DISCOVERY_END_YEAR}')
print(f'Contract month codes: {list(CONTRACT_MONTH_CODES.keys())}')
print(f'Tradeable only: {TRADEABLE_ONLY}')
print(f'Instruments parquet: {INSTRUMENTS_PARQUET_PATH}')
print(f'Contract bars output dir: {BARS_OUTPUT_DIR}')
print(f'Continuous bars output dir: {CONTINUOUS_BARS_OUTPUT_DIR}')
print(f'Bar window: {BAR_START.isoformat()} -> {BAR_END.isoformat()}')
print(f'Max history pages per instrument/day: {BAR_MAX_REQUEST_PAGES}')
print(
    'Backtest settings: '
    f'window={BACKTEST_START.isoformat()} -> {BACKTEST_END.isoformat()} '
    f'fast={BACKTEST_FAST_EMA_PERIOD} slow={BACKTEST_SLOW_EMA_PERIOD} '
    f'trade_size={BACKTEST_TRADE_SIZE} starting_balance={BACKTEST_STARTING_BALANCE}'
)


Profile: Apex
Root product: MNQ
Root exchange: CME
Discovery years: 2019 -> 2027
Contract month codes: ['F', 'G', 'H', 'J', 'K', 'M', 'N', 'Q', 'U', 'V', 'X', 'Z']
Tradeable only: False
Instruments parquet: /Users/agedvagabond/Developer/nautilus_trader_env/data/rithmic/mnq_futures_contracts.parquet
Contract bars output dir: /Users/agedvagabond/Developer/nautilus_trader_env/data/rithmic/mnq_bars_1minute_by_day
Continuous bars output dir: /Users/agedvagabond/Developer/nautilus_trader_env/data/rithmic/mnq_continuous_bars_1minute_by_day
Bar window: 2024-01-01T00:00:00+00:00 -> 2026-03-28T06:32:00+00:00
Max history pages per instrument/day: 64
Backtest settings: window=2024-01-01T00:00:00+00:00 -> 2026-03-28T06:32:00+00:00 fast=10 slow=20 trade_size=1 starting_balance=250000.0


## Step 3: Define helper functions

What this cell does:
- Defines the gateway builder, contract-request builder, parquet read/write helpers, preview helpers, and shared time utilities.
- Defines the helper class used to convert native Rithmic bar responses into Nautilus `Bar` objects.
- Defines the helper that creates the extra synthetic root instrument used by the optional continuous-history workflow, including a max expiration timestamp for long backtests.

What to expect:
- No meaningful output.
- If this cell fails, the later discovery and bar-download cells will not work.


In [9]:
def build_gateway(config: RithmicDataClientConfig, enable_history: bool) -> RithmicGateway:
    return RithmicGateway(
        environment=to_binding_environment(config.environment),
        username=config.username,
        password=config.password,
        system_name=config.system_name,
        app_name=config.app_name,
        app_version=config.app_version,
        fcm_id=config.fcm_id or '',
        ib_id=config.ib_id or '',
        account_id='',
        server=config.server,
        alt_server=config.alt_server,
        enable_ticker=True,
        enable_order=False,
        enable_pnl=False,
        enable_history=enable_history,
    )


def build_contract_requests(
    product_code: str,
    start_year: int,
    end_year: int,
    month_codes: dict[str, int],
) -> list[dict[str, object]]:
    requests: list[dict[str, object]] = []
    for year in range(start_year, end_year + 1):
        for month_code, month_number in month_codes.items():
            requests.append(
                {
                    'contract_year': year,
                    'contract_month_code': month_code,
                    'contract_month': month_number,
                    'symbol_candidates': [
                        f'{product_code}{month_code}{year % 10}',
                        f'{product_code}{month_code}{year % 100:02d}',
                    ],
                }
            )
    return requests


def format_ts_ns(value: int | None) -> str | None:
    if not value:
        return None
    return datetime.fromtimestamp(value / 1_000_000_000, tz=UTC).isoformat()


def read_nautilus_objects(path: Path, data_cls: type):
    if not path.exists():
        return []
    table = pq.read_table(path)
    objects = list(ArrowSerializer.deserialize(data_cls=data_cls, batch=table))
    if data_cls is Bar and objects and not isinstance(objects[0], Bar):
        return Bar.from_pyo3_list(objects)
    return objects


def write_nautilus_objects(
    path: Path,
    objects: list,
    data_cls: type,
    key_fn,
    merge_existing: bool = True,
):
    merged: dict[str, object] = {}
    if merge_existing and path.exists():
        for obj in read_nautilus_objects(path, data_cls):
            merged[key_fn(obj)] = obj
    for obj in objects:
        merged[key_fn(obj)] = obj
    ordered_keys = sorted(merged)
    ordered_objects = [merged[key] for key in ordered_keys]
    if not ordered_objects:
        return []
    path.parent.mkdir(parents=True, exist_ok=True)
    table = ArrowSerializer.serialize_batch(ordered_objects, data_cls=data_cls)
    pq.write_table(table, where=path)
    return ordered_objects


def instrument_key(instrument: FuturesContract) -> str:
    return str(instrument.id)


def bar_key(bar: Bar) -> str:
    return f'{bar.bar_type}|{bar.ts_event}'


def is_continuous_root_instrument(instrument: FuturesContract) -> bool:
    info = getattr(instrument, 'info', {}) or {}
    return bool(info.get('is_continuous_root'))


def select_continuous_template_instrument(
    instruments: list[FuturesContract],
    summary_df: pd.DataFrame,
) -> FuturesContract:
    by_symbol = {
        getattr(getattr(instrument, 'raw_symbol', None), 'value', None): instrument
        for instrument in instruments
    }
    if not summary_df.empty:
        ok_symbols = summary_df.loc[
            summary_df['status'] == 'ok',
            'resolved_symbol',
        ].dropna().tolist()
        for symbol in ok_symbols:
            instrument = by_symbol.get(symbol)
            if instrument is not None:
                return instrument
    return instruments[0]


def build_continuous_root_instrument(
    template_instrument: FuturesContract,
    root_symbol: str,
) -> FuturesContract:
    values = FuturesContract.to_dict(template_instrument)
    template_id = str(template_instrument.id)
    _, _, venue = template_id.partition('.')
    if not venue:
        raise ValueError(f'Could not derive venue from template instrument id: {template_id}')

    info = dict(values.get('info') or {})
    info['description'] = root_symbol
    info['is_continuous_root'] = True
    info['continuous_template_instrument_id'] = template_id
    info['continuous_template_raw_symbol'] = getattr(
        getattr(template_instrument, 'raw_symbol', None),
        'value',
        None,
    )
    info['is_backtest_only'] = True
    info['continuous_history_mode'] = 'root_symbol'

    values['id'] = f'{root_symbol}.{venue}'
    values['expiration_ns'] = pd.Timestamp.max.tz_localize('UTC').value
    values['raw_symbol'] = root_symbol
    values['info'] = info
    return FuturesContract.from_dict(values)


def instruments_to_frame(instruments: list[FuturesContract]) -> pd.DataFrame:
    rows = []
    for instrument in instruments:
        info = getattr(instrument, 'info', {}) or {}
        rows.append(
            {
                'instrument_id': str(instrument.id),
                'symbol': getattr(getattr(instrument, 'raw_symbol', None), 'value', None),
                'exchange': getattr(instrument, 'exchange', None),
                'product_code': info.get('product_code'),
                'description': info.get('description'),
                'is_tradeable': info.get('is_tradeable'),
                'is_continuous_root': info.get('is_continuous_root', False),
                'continuous_template_instrument_id': info.get('continuous_template_instrument_id'),
                'underlying': getattr(instrument, 'underlying', None),
                'currency': getattr(getattr(instrument, 'currency', None), 'code', None),
                'price_precision': getattr(instrument, 'price_precision', None),
                'price_increment': str(getattr(instrument, 'price_increment', '')),
                'multiplier': str(getattr(instrument, 'multiplier', '')),
                'lot_size': str(getattr(instrument, 'lot_size', '')),
                'expiration_utc': format_ts_ns(getattr(instrument, 'expiration_ns', None)),
            }
        )
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(['exchange', 'product_code', 'symbol'], kind='stable').reset_index(drop=True)


def bars_to_frame(bars: list[Bar]) -> pd.DataFrame:
    rows = []
    for bar in bars:
        rows.append(
            {
                'bar_type': str(bar.bar_type),
                'instrument_id': str(bar.bar_type.instrument_id),
                'ts_event_utc': format_ts_ns(bar.ts_event),
                'open': float(bar.open),
                'high': float(bar.high),
                'low': float(bar.low),
                'close': float(bar.close),
                'volume': float(bar.volume),
            }
        )
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values(['instrument_id', 'ts_event_utc'], kind='stable').reset_index(drop=True)


class NotebookLogger:
    def warning(self, message: str) -> None:
        print(f'[rithmic-notebook][warning] {message}')


class NotebookBarConverter:
    _convert_time_bars = RithmicLiveDataClient._convert_time_bars
    _bar_timestamp_from_response = RithmicLiveDataClient._bar_timestamp_from_response
    _fallback_bar_timestamp = RithmicLiveDataClient._fallback_bar_timestamp

    def __init__(self, instruments: list[FuturesContract]) -> None:
        self._by_id = {instrument.id: instrument for instrument in instruments}
        self._log = NotebookLogger()

    def _lookup_instrument(self, instrument_id):
        return self._by_id.get(instrument_id)


def minute_bar_type(instrument: FuturesContract) -> BarType:
    return BarType.from_str(f'{instrument.id}-1-MINUTE-LAST-EXTERNAL')


def normalize_utc_timestamp(value) -> pd.Timestamp:
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        ts = ts.tz_localize('UTC')
    else:
        ts = ts.tz_convert('UTC')
    return ts


def iter_utc_days(start, end):
    current = normalize_utc_timestamp(start).floor('D')
    stop = normalize_utc_timestamp(end)
    while current < stop:
        yield current
        current += pd.Timedelta(days=1)


## Step 4: Build the contract probe list

What this cell does:
- Creates the symbol candidates that will be probed for contract discovery.
- Uses the selected root, the configured year range, and the configured Rithmic month codes.

What to expect:
- A one-row table showing the selected root.
- A preview table of contract years, month codes, and symbol candidates such as `MNQH4` and `MNQH24`.
- Review this table before running discovery.


In [10]:
selected_products = [
    {
        'product_code': ROOT_PRODUCT,
        'exchange': ROOT_EXCHANGE,
        'group': 'User-selected root',
    }
]
contract_requests = build_contract_requests(
    product_code=ROOT_PRODUCT,
    start_year=DISCOVERY_START_YEAR,
    end_year=DISCOVERY_END_YEAR,
    month_codes=CONTRACT_MONTH_CODES,
)
request_preview_df = pd.DataFrame(
    {
        'contract_year': request['contract_year'],
        'contract_month_code': request['contract_month_code'],
        'contract_month': request['contract_month'],
        'symbol_candidates': ', '.join(request['symbol_candidates']),
    }
    for request in contract_requests
)
print('Using the user-selected root configuration:')
display(pd.DataFrame(selected_products))
print(f'Planned contract symbol probes: {len(contract_requests)}')
display(request_preview_df)


Using the user-selected root configuration:


,product_code,exchange,group
0,MNQ,CME,User-selected root


Planned contract symbol probes: 108


,contract_year,contract_month_code,contract_month,symbol_candidates
0,2019,F,1,"MNQF9, MNQF19"
1,2019,G,2,"MNQG9, MNQG19"
2,2019,H,3,"MNQH9, MNQH19"
3,2019,J,4,"MNQJ9, MNQJ19"
4,2019,K,5,"MNQK9, MNQK19"
...,...,...,...,...
103,2027,Q,8,"MNQQ7, MNQQ27"
104,2027,U,9,"MNQU7, MNQU27"
105,2027,V,10,"MNQV7, MNQV27"
106,2027,X,11,"MNQX7, MNQX27"


## Step 5: Discover and persist instruments

After this cell you will see that rithmic does not provide a long history for individual contracts. As a result we can skip step 8 and jump straight to 9.

What this cell does:
- Connects to Rithmic.
- Probes each candidate symbol until one resolves for that contract slot.
- Converts successful results into Nautilus `FuturesContract` objects.
- Creates one extra synthetic root instrument by copying the first successful real contract, renaming it to the root symbol, and setting its expiration to the maximum pandas timestamp.
- Merges and writes those instruments to the configured parquet path.

What to expect:
- `discovery_summary_df` showing which contract probes succeeded or failed.
- A preview of the discovered Nautilus instruments, including the extra backtest-only root instrument copy.
- If zero instruments are found, the cell raises after showing the summary so you can inspect the failures.
- You may not see the files appear in your IDE, check the OUTPUT_DIR path configured in cell 2, the files will be there assuming permissions are correct


In [16]:
async def fetch_available_instruments(
    product_code: str,
    exchange: str,
    requests: list[dict[str, object]],
    profile: str | None = None,
    tradeable_only: bool = False,
) -> tuple[list[FuturesContract], pd.DataFrame]:
    config = RithmicDataClientConfig.from_env(profile=profile)
    gateway = build_gateway(config, enable_history=False)
    provider = RustRithmicInstrumentProvider(gateway)
    converter = PythonRithmicInstrumentProvider(config)

    instruments_by_id: dict[str, FuturesContract] = {}
    summary_rows: list[dict[str, object]] = []
    connected = False

    try:
        await gateway.connect()
        connected = True

        for request in requests:
            loaded = None
            resolved_symbol = None
            last_error: str | None = None

            for symbol in request['symbol_candidates']:
                requested_symbol = str(symbol)
                try:
                    candidate = await provider.load_instrument_async(requested_symbol, exchange)
                except Exception as exc:
                    last_error = f'{type(exc).__name__}: {exc}'
                    continue

                if tradeable_only and not candidate.is_tradeable:
                    last_error = 'Instrument returned by Rithmic but filtered out as non-tradeable'
                    continue

                loaded = candidate
                resolved_symbol = requested_symbol
                break

            if loaded is None:
                summary_rows.append(
                    {
                        'exchange': exchange,
                        'product_code': product_code,
                        'contract_year': request['contract_year'],
                        'contract_month_code': request['contract_month_code'],
                        'contract_month': request['contract_month'],
                        'requested_symbol': None,
                        'resolved_symbol': None,
                        'matched_contracts': 0,
                        'status': 'missing',
                        'error': last_error,
                    }
                )
                continue

            instrument = converter._convert_instrument(loaded)
            instruments_by_id[instrument_key(instrument)] = instrument
            summary_rows.append(
                {
                    'exchange': exchange,
                    'product_code': product_code,
                    'contract_year': request['contract_year'],
                    'contract_month_code': request['contract_month_code'],
                    'contract_month': request['contract_month'],
                    'requested_symbol': resolved_symbol,
                    'resolved_symbol': loaded.symbol,
                    'matched_contracts': 1,
                    'status': 'ok',
                    'error': None,
                }
            )
    finally:
        if connected:
            await gateway.disconnect()

    instruments = [instruments_by_id[key] for key in sorted(instruments_by_id)]
    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(
            ['contract_year', 'contract_month'],
            kind='stable',
        ).reset_index(drop=True)
    return instruments, summary_df


discovered_instruments, discovery_summary_df = await fetch_available_instruments(
    product_code=ROOT_PRODUCT,
    exchange=ROOT_EXCHANGE,
    requests=contract_requests,
    profile=PROFILE,
    tradeable_only=TRADEABLE_ONLY,
)

print(f'Discovered contracts this run: {len(discovered_instruments)}')
display(discovery_summary_df)

if not discovered_instruments:
    raise RuntimeError(
        'No Rithmic contracts were discovered for the selected product. '
        'Inspect discovery_summary_df for the per-contract probe errors.'
    )

continuous_template_instrument = select_continuous_template_instrument(
    discovered_instruments,
    discovery_summary_df,
)
continuous_root_instrument = build_continuous_root_instrument(
    template_instrument=continuous_template_instrument,
    root_symbol=ROOT_PRODUCT,
)

print(
    'Added synthetic backtest-only root instrument copy: '
    f'{continuous_root_instrument.id} derived from {continuous_template_instrument.id}'
)

stored_instruments = write_nautilus_objects(
    path=INSTRUMENTS_PARQUET_PATH,
    objects=[*discovered_instruments, continuous_root_instrument],
    data_cls=FuturesContract,
    key_fn=instrument_key,
    merge_existing=MERGE_WITH_EXISTING_INSTRUMENTS,
)

stored_instruments_df = instruments_to_frame(stored_instruments)
stored_contract_count = int((~stored_instruments_df['is_continuous_root'].fillna(False)).sum())
stored_root_count = int(stored_instruments_df['is_continuous_root'].fillna(False).sum())

print(f'Contracts stored in parquet: {stored_contract_count}')
print(f'Continuous root instrument copies stored in parquet: {stored_root_count}')
print(f'Instrument parquet path: {INSTRUMENTS_PARQUET_PATH}')
display(stored_instruments_df.head(100))


RITHMIC CONFIG >>> env=Demo system=Apex account= fcm=Apex ib=Apex user=APEX-3396 url=wss://rprotocol.rithmic.com:443 alt=wss://rprotocol.rithmic.com:443
Discovered contracts this run: 5


,exchange,product_code,contract_year,contract_month_code,contract_month,requested_symbol,resolved_symbol,matched_contracts,status,error
0,CME,MNQ,2019,F,1,None,None,0,missing,RuntimeError: Instrument error: Reference data...
1,CME,MNQ,2019,G,2,None,None,0,missing,RuntimeError: Instrument error: Reference data...
2,CME,MNQ,2019,H,3,None,None,0,missing,RuntimeError: Instrument error: Reference data...
3,CME,MNQ,2019,J,4,None,None,0,missing,RuntimeError: Instrument error: Reference data...
4,CME,MNQ,2019,K,5,None,None,0,missing,RuntimeError: Instrument error: Reference data...
...,...,...,...,...,...,...,...,...,...,...
103,CME,MNQ,2027,Q,8,None,None,0,missing,RuntimeError: Instrument error: Reference data...
104,CME,MNQ,2027,U,9,None,None,0,missing,RuntimeError: Instrument error: Reference data...
105,CME,MNQ,2027,V,10,None,None,0,missing,RuntimeError: Instrument error: Reference data...
106,CME,MNQ,2027,X,11,None,None,0,missing,RuntimeError: Instrument error: Reference data...


Added synthetic backtest-only root instrument copy: MNQ.CME.RITHMIC derived from MNQM6.CME.RITHMIC
Contracts stored in parquet: 5
Continuous root instrument copies stored in parquet: 1
Instrument parquet path: /Users/agedvagabond/Developer/nautilus_trader_env/data/rithmic/mnq_futures_contracts.parquet


,instrument_id,symbol,exchange,product_code,description,is_tradeable,is_continuous_root,continuous_template_instrument_id,underlying,currency,price_precision,price_increment,multiplier,lot_size,expiration_utc
0,MNQ.CME.RITHMIC,MNQ,CME,MNQ,MNQ,True,True,MNQM6.CME.RITHMIC,MNQ,None,2,0.25,2.0,1.0,2262-04-11T23:47:16.854776+00:00
1,MNQH7.CME.RITHMIC,MNQH7,CME,MNQ,Micro E-mini Nasdaq-100,True,False,None,MNQ,None,2,0.25,2.0,1.0,2027-03-19T00:00:00+00:00
2,MNQM6.CME.RITHMIC,MNQM6,CME,MNQ,Micro E-mini Nasdaq-100,True,False,None,MNQ,None,2,0.25,2.0,1.0,2026-06-18T00:00:00+00:00
3,MNQM7.CME.RITHMIC,MNQM7,CME,MNQ,Micro E-mini Nasdaq-100,True,False,None,MNQ,None,2,0.25,2.0,1.0,2027-06-17T00:00:00+00:00
4,MNQU6.CME.RITHMIC,MNQU6,CME,MNQ,Micro E-mini Nasdaq-100,True,False,None,MNQ,None,2,0.25,2.0,1.0,2026-09-18T00:00:00+00:00
5,MNQZ6.CME.RITHMIC,MNQZ6,CME,MNQ,Micro E-mini Nasdaq-100,True,False,None,MNQ,None,2,0.25,2.0,1.0,2026-12-18T00:00:00+00:00


## Step 6: Prepare the bar-download workflows

What this section covers:
- The next helper cell defines paginated daily 1-minute bar downloads.
- The notebook supports both optional specific-contract history and continuous root-symbol history (needed to run the final test cell).
- The notebook resumes requests from the last returned bar timestamp because Rithmic history can truncate long responses.
- Bar data is written as Nautilus `Bar` parquet, one UTC day per file.
- Each instrument/day batch is flushed to parquet immediately after download so the notebook does not retain a full day of bars in memory.

What to expect later:
- A per-day summary table for whichever download cell you run.
- Optional error rows when a request stalls, exhausts paging, or fails.


## Step 7: Define paginated bar-download helpers

What this cell does:
- Loads persisted instruments back from parquet.
- Separates real contract instruments from the extra synthetic root instrument.
- Defines the per-request paged bar loop shared by the contract and continuous-root workflows.
- Defines the per-day writers that merge and deduplicate Nautilus `Bar` objects.
- Defines the helper used by the final EMA-crossover cell to load the continuous daily parquet files back into memory.
- Saves each downloaded instrument/day batch to parquet immediately so memory is released before the next request.

What to expect:
- No meaningful output.
- This cell only defines logic; it does not contact Rithmic yet.


In [11]:
def load_contract_instruments_for_bars(
    path: Path,
    instrument_limit: int | None = None,
) -> list[FuturesContract]:
    instruments = [
        instrument
        for instrument in read_nautilus_objects(path, FuturesContract)
        if not is_continuous_root_instrument(instrument)
    ]
    if instrument_limit is not None:
        instruments = instruments[:instrument_limit]
    return instruments


def load_continuous_root_instrument(
    path: Path,
    root_symbol: str,
    exchange: str,
) -> FuturesContract:
    instruments = read_nautilus_objects(path, FuturesContract)
    for instrument in instruments:
        symbol = getattr(getattr(instrument, 'raw_symbol', None), 'value', None)
        instrument_exchange = getattr(instrument, 'exchange', None)
        if (
            symbol == root_symbol
            and instrument_exchange == exchange
            and is_continuous_root_instrument(instrument)
        ):
            return instrument
    raise RuntimeError(
        'Could not find the synthetic continuous root instrument in the instrument parquet. '
        'Run the discovery cell first.'
    )


def ts_ns_to_utc(ts_ns: int) -> pd.Timestamp:
    return pd.Timestamp(ts_ns, tz='UTC')


async def request_minute_bars_with_resume(
    client: RithmicDataClient,
    converter: NotebookBarConverter,
    request_symbol: str,
    request_exchange: str,
    target_instrument: FuturesContract,
    start,
    end,
    max_request_pages: int,
) -> tuple[list[Bar], int, str]:
    start_ts = normalize_utc_timestamp(start)
    end_ts = normalize_utc_timestamp(end)
    cursor_ts = start_ts
    end_ts_ns = end_ts.value
    one_minute_ns = 60 * 1_000_000_000

    if not request_symbol or not request_exchange:
        raise ValueError('request_symbol and request_exchange are required for bar downloads')

    bars_by_key: dict[str, Bar] = {}
    request_pages = 0

    while cursor_ts < end_ts and request_pages < max_request_pages:
        request_pages += 1
        responses = await client.request_bars(
            request_symbol,
            request_exchange,
            'MinuteBar',
            1,
            int(cursor_ts.timestamp()),
            int(end_ts.timestamp()),
        )
        converted = converter._convert_time_bars(
            minute_bar_type(target_instrument),
            responses,
            target_instrument.id,
        )

        if not converted:
            status = 'empty' if request_pages == 1 and not bars_by_key else 'stalled_empty'
            ordered = [bars_by_key[key] for key in sorted(bars_by_key)]
            return ordered, request_pages, status

        for bar in converted:
            bars_by_key[bar_key(bar)] = bar

        last_bar_ts_ns = max(bar.ts_event for bar in converted)
        if last_bar_ts_ns >= end_ts_ns - one_minute_ns:
            ordered = [bars_by_key[key] for key in sorted(bars_by_key)]
            return ordered, request_pages, 'complete'

        next_cursor_ts = ts_ns_to_utc(last_bar_ts_ns)
        if next_cursor_ts <= cursor_ts:
            ordered = [bars_by_key[key] for key in sorted(bars_by_key)]
            return ordered, request_pages, 'stalled_cursor'

        cursor_ts = next_cursor_ts

    ordered = [bars_by_key[key] for key in sorted(bars_by_key)]
    if cursor_ts >= end_ts:
        return ordered, request_pages, 'complete'
    if request_pages >= max_request_pages:
        return ordered, request_pages, 'max_pages_exhausted'
    return ordered, request_pages, 'complete'


async def download_minute_bars_by_day(
    instruments_path: Path,
    output_dir: Path,
    start,
    end,
    profile: str | None = None,
    instrument_limit: int | None = None,
    day_limit: int | None = None,
    max_request_pages: int = 64,
    merge_existing: bool = True,
    skip_existing_files: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    instruments = load_contract_instruments_for_bars(
        path=instruments_path,
        instrument_limit=instrument_limit,
    )
    if not instruments:
        raise RuntimeError('No contract instruments available for the requested bar download scope.')

    start_ts = normalize_utc_timestamp(start)
    end_ts = normalize_utc_timestamp(end)
    if start_ts >= end_ts:
        raise ValueError('BAR_START must be earlier than BAR_END.')

    config = RithmicDataClientConfig.from_env(profile=profile)
    gateway = build_gateway(config, enable_history=True)
    client = RithmicDataClient(gateway)
    converter = NotebookBarConverter(instruments)
    summary_rows: list[dict[str, object]] = []
    error_rows: list[dict[str, object]] = []
    connected = False

    try:
        await gateway.connect()
        connected = True

        for day_index, day_start in enumerate(iter_utc_days(start_ts, end_ts), start=1):
            if day_limit is not None and day_index > day_limit:
                break

            day_end = min(day_start + pd.Timedelta(days=1), end_ts)
            day_path = output_dir / f'{day_start:%Y-%m-%d}.parquet'

            if skip_existing_files and day_path.exists():
                summary_rows.append(
                    {
                        'date_utc': f'{day_start:%Y-%m-%d}',
                        'path': str(day_path),
                        'symbols_requested': len(instruments),
                        'symbols_with_data': None,
                        'request_pages': 0,
                        'resumed_instruments': 0,
                        'bars_written_this_run': 0,
                        'bars_stored_total': len(read_nautilus_objects(day_path, Bar)),
                        'status': 'skipped_existing',
                    }
                )
                continue

            symbols_with_data = 0
            request_pages = 0
            resumed_instruments = 0
            bars_written_this_run = 0

            for instrument in instruments:
                symbol = getattr(getattr(instrument, 'raw_symbol', None), 'value', None)
                exchange = getattr(instrument, 'exchange', None)
                if not symbol or not exchange:
                    error_rows.append(
                        {
                            'date_utc': f'{day_start:%Y-%m-%d}',
                            'instrument_id': str(instrument.id),
                            'symbol': symbol,
                            'exchange': exchange,
                            'error': 'Instrument is missing raw_symbol or exchange',
                        }
                    )
                    continue

                try:
                    bars, instrument_pages, page_status = await request_minute_bars_with_resume(
                        client=client,
                        converter=converter,
                        request_symbol=symbol,
                        request_exchange=exchange,
                        target_instrument=instrument,
                        start=day_start,
                        end=day_end,
                        max_request_pages=max_request_pages,
                    )
                except Exception as exc:
                    error_rows.append(
                        {
                            'date_utc': f'{day_start:%Y-%m-%d}',
                            'instrument_id': str(instrument.id),
                            'symbol': symbol,
                            'exchange': exchange,
                            'error': f'{type(exc).__name__}: {exc}',
                        }
                    )
                    continue

                request_pages += instrument_pages
                if instrument_pages > 1:
                    resumed_instruments += 1

                if page_status in {'stalled_empty', 'stalled_cursor', 'max_pages_exhausted'}:
                    error_rows.append(
                        {
                            'date_utc': f'{day_start:%Y-%m-%d}',
                            'instrument_id': str(instrument.id),
                            'symbol': symbol,
                            'exchange': exchange,
                            'error': (
                                f'Paged history request ended with status={page_status} '
                                f'after {instrument_pages} request(s)'
                            ),
                        }
                    )

                if bars:
                    symbols_with_data += 1
                    write_nautilus_objects(
                        path=day_path,
                        objects=bars,
                        data_cls=Bar,
                        key_fn=bar_key,
                        merge_existing=merge_existing,
                    )
                    bars_written_this_run += len(bars)
                    del bars

            bars_stored_total = len(read_nautilus_objects(day_path, Bar)) if day_path.exists() else 0
            status = 'written' if bars_written_this_run > 0 else 'empty'

            summary_rows.append(
                {
                    'date_utc': f'{day_start:%Y-%m-%d}',
                    'path': str(day_path),
                    'symbols_requested': len(instruments),
                    'symbols_with_data': symbols_with_data,
                    'request_pages': request_pages,
                    'resumed_instruments': resumed_instruments,
                    'bars_written_this_run': bars_written_this_run,
                    'bars_stored_total': bars_stored_total,
                    'status': status,
                }
            )
    finally:
        if connected:
            await gateway.disconnect()

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(['date_utc'], kind='stable').reset_index(drop=True)
    errors_df = pd.DataFrame(error_rows)
    if not errors_df.empty:
        errors_df = errors_df.sort_values(
            ['date_utc', 'instrument_id'],
            kind='stable',
        ).reset_index(drop=True)
    return summary_df, errors_df



async def download_continuous_root_bars_by_day(
    instruments_path: Path,
    output_dir: Path,
    root_symbol: str,
    exchange: str,
    start,
    end,
    profile: str | None = None,
    day_limit: int | None = None,
    max_request_pages: int = 64,
    merge_existing: bool = True,
    skip_existing_files: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    root_instrument = load_continuous_root_instrument(
        path=instruments_path,
        root_symbol=root_symbol,
        exchange=exchange,
    )

    start_ts = normalize_utc_timestamp(start)
    end_ts = normalize_utc_timestamp(end)
    if start_ts >= end_ts:
        raise ValueError('BAR_START must be earlier than BAR_END.')

    config = RithmicDataClientConfig.from_env(profile=profile)
    gateway = build_gateway(config, enable_history=True)
    client = RithmicDataClient(gateway)
    converter = NotebookBarConverter([root_instrument])
    summary_rows: list[dict[str, object]] = []
    error_rows: list[dict[str, object]] = []
    connected = False

    try:
        await gateway.connect()
        connected = True

        for day_index, day_start in enumerate(iter_utc_days(start_ts, end_ts), start=1):
            if day_limit is not None and day_index > day_limit:
                break

            day_end = min(day_start + pd.Timedelta(days=1), end_ts)
            day_path = output_dir / f'{day_start:%Y-%m-%d}.parquet'

            if skip_existing_files and day_path.exists():
                summary_rows.append(
                    {
                        'date_utc': f'{day_start:%Y-%m-%d}',
                        'path': str(day_path),
                        'request_symbol': root_symbol,
                        'request_pages': 0,
                        'resumed': False,
                        'bars_written_this_run': 0,
                        'bars_stored_total': len(read_nautilus_objects(day_path, Bar)),
                        'status': 'skipped_existing',
                    }
                )
                continue

            try:
                bars, request_pages, page_status = await request_minute_bars_with_resume(
                    client=client,
                    converter=converter,
                    request_symbol=root_symbol,
                    request_exchange=exchange,
                    target_instrument=root_instrument,
                    start=day_start,
                    end=day_end,
                    max_request_pages=max_request_pages,
                )
            except Exception as exc:
                error_rows.append(
                    {
                        'date_utc': f'{day_start:%Y-%m-%d}',
                        'instrument_id': str(root_instrument.id),
                        'symbol': root_symbol,
                        'exchange': exchange,
                        'error': f'{type(exc).__name__}: {exc}',
                    }
                )
                continue

            if page_status in {'stalled_empty', 'stalled_cursor', 'max_pages_exhausted'}:
                error_rows.append(
                    {
                        'date_utc': f'{day_start:%Y-%m-%d}',
                        'instrument_id': str(root_instrument.id),
                        'symbol': root_symbol,
                        'exchange': exchange,
                        'error': (
                            f'Paged history request ended with status={page_status} '
                            f'after {request_pages} request(s)'
                        ),
                    }
                )

            bars_written_this_run = 0
            if bars:
                write_nautilus_objects(
                    path=day_path,
                    objects=bars,
                    data_cls=Bar,
                    key_fn=bar_key,
                    merge_existing=merge_existing,
                )
                bars_written_this_run = len(bars)
                del bars

            bars_stored_total = len(read_nautilus_objects(day_path, Bar)) if day_path.exists() else 0
            summary_rows.append(
                {
                    'date_utc': f'{day_start:%Y-%m-%d}',
                    'path': str(day_path),
                    'request_symbol': root_symbol,
                    'request_pages': request_pages,
                    'resumed': request_pages > 1,
                    'bars_written_this_run': bars_written_this_run,
                    'bars_stored_total': bars_stored_total,
                    'status': 'written' if bars_written_this_run > 0 else 'empty',
                }
            )
    finally:
        if connected:
            await gateway.disconnect()

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(['date_utc'], kind='stable').reset_index(drop=True)
    errors_df = pd.DataFrame(error_rows)
    if not errors_df.empty:
        errors_df = errors_df.sort_values(
            ['date_utc', 'instrument_id'],
            kind='stable',
        ).reset_index(drop=True)
    return summary_df, errors_df



def load_bars_from_daily_directory(
    output_dir: Path,
    instrument_id: str | None = None,
    start=None,
    end=None,
) -> list[Bar]:
    if not output_dir.exists():
        return []

    start_ts = normalize_utc_timestamp(start) if start is not None else None
    end_ts = normalize_utc_timestamp(end) if end is not None else None
    bars_by_id: dict[str, Bar] = {}

    for path in sorted(output_dir.glob('*.parquet')):
        try:
            day_start = pd.Timestamp(path.stem, tz='UTC')
        except Exception:
            day_start = None

        if start_ts is not None and day_start is not None and day_start + pd.Timedelta(days=1) <= start_ts:
            continue
        if end_ts is not None and day_start is not None and day_start >= end_ts:
            continue

        for bar in read_nautilus_objects(path, Bar):
            if instrument_id is not None and str(bar.bar_type.instrument_id) != instrument_id:
                continue
            if start_ts is not None and bar.ts_event < start_ts.value:
                continue
            if end_ts is not None and bar.ts_event >= end_ts.value:
                continue
            bars_by_id[bar_key(bar)] = bar

    return [bars_by_id[key] for key in sorted(bars_by_id)]


## Step 8: Run the contract-specific 1-minute bar download (Optional)

What this cell does:
- Reads the discovered contract parquet.
- Ignores the extra synthetic root instrument.
- Downloads 1-minute bars for each discovered real contract over the configured bar window.
- Writes one Nautilus parquet file per UTC day.
- Flushes each downloaded instrument/day batch to parquet immediately after download so memory is released before the next request.

What to expect:
- `bar_summary_df` with one row per UTC day touched.
- `bar_errors_df` if any contract/day requests failed or stalled.
- A preview of the first written daily bar file when available.
- You may not see the files appear in your IDE, check the OUTPUT_DIR path configured in cell 2, the files will be there assuming permissions are correct


In [ ]:
bar_summary_df, bar_errors_df = await download_minute_bars_by_day(
    instruments_path=INSTRUMENTS_PARQUET_PATH,
    output_dir=BARS_OUTPUT_DIR,
    start=BAR_START,
    end=BAR_END,
    profile=PROFILE,
    instrument_limit=BAR_INSTRUMENT_LIMIT,
    day_limit=BAR_DAY_LIMIT,
    max_request_pages=BAR_MAX_REQUEST_PAGES,
    merge_existing=MERGE_WITH_EXISTING_BARS,
    skip_existing_files=SKIP_EXISTING_BAR_FILES,
)

print(f'Contract daily bar files touched: {len(bar_summary_df)}')
display(bar_summary_df)

if not bar_errors_df.empty:
    print(f'Contract bar download errors: {len(bar_errors_df)}')
    display(bar_errors_df.head(100))

written_paths = [
    Path(path)
    for path in bar_summary_df.loc[bar_summary_df['status'] == 'written', 'path'].tolist()
]
if written_paths:
    sample_bars = read_nautilus_objects(written_paths[0], Bar)
    print(f'Sample contract day file: {written_paths[0]}')
    display(bars_to_frame(sample_bars).head(50))
else:
    print('No contract daily bar parquet files were written in this run.')


## Step 9: root-symbol continuous 1-minute bar download

It is possible to download a much larger historical dataset of history as front month adjusted data using the root symbol.

What this cell does:
- Reads the extra synthetic root instrument written during discovery.
- Requests 1-minute history from Rithmic using the root symbol itself, such as `MNQ`.
- Saves those bars against the synthetic root instrument id, such as `MNQ.CME`.
- Uses that synthetic instrument only as a backtest placeholder because specific-contract history is shallower than the root-symbol history.
- Writes one Nautilus parquet file per UTC day to `CONTINUOUS_BARS_OUTPUT_DIR`.

What to expect:
- `continuous_bar_summary_df` with one row per UTC day touched.
- `continuous_bar_errors_df` if the root request fails, stalls, or exhausts paging.
- A preview of the first written continuous daily bar file when available.
- You may not see the files appear in your IDE, check the OUTPUT_DIR path configured in cell 2, the files will be there assuming permissions are correct

Use this optional cell when you want a longer continuous history series for backtesting  the specific-contract files. In this workflow the continuous root series is expected to extend back to 2019-06.


In [24]:
continuous_bar_summary_df, continuous_bar_errors_df = await download_continuous_root_bars_by_day(
    instruments_path=INSTRUMENTS_PARQUET_PATH,
    output_dir=CONTINUOUS_BARS_OUTPUT_DIR,
    root_symbol=ROOT_PRODUCT,
    exchange=ROOT_EXCHANGE,
    start=BAR_START,
    end=BAR_END,
    profile=PROFILE,
    day_limit=BAR_DAY_LIMIT,
    max_request_pages=BAR_MAX_REQUEST_PAGES,
    merge_existing=MERGE_WITH_EXISTING_BARS,
    skip_existing_files=SKIP_EXISTING_BAR_FILES,
)

print(f'Continuous daily bar files touched: {len(continuous_bar_summary_df)}')
display(continuous_bar_summary_df)

if not continuous_bar_errors_df.empty:
    print(f'Continuous bar download errors: {len(continuous_bar_errors_df)}')
    display(continuous_bar_errors_df.head(100))

continuous_written_paths = [
    Path(path)
    for path in continuous_bar_summary_df.loc[
        continuous_bar_summary_df['status'] == 'written',
        'path',
    ].tolist()
]
if continuous_written_paths:
    sample_bars = read_nautilus_objects(continuous_written_paths[0], Bar)
    print(f'Sample continuous day file: {continuous_written_paths[0]}')
    display(bars_to_frame(sample_bars).head(50))
else:
    print('No continuous daily bar parquet files were written in this run.')


RITHMIC CONFIG >>> env=Demo system=Apex account= fcm=Apex ib=Apex user=APEX-3396 url=wss://rprotocol.rithmic.com:443 alt=wss://rprotocol.rithmic.com:443


CancelledError: 

## Step 10: Run a simple EMA-crossover backtest on the continuous root data

What this cell does:
- Loads the synthetic continuous root instrument from the instrument parquet.
- Loads the already-downloaded continuous daily bar files back into memory for the configured backtest window.
- Runs Nautilus' built-in `EMACross` example strategy against that continuous series only.
- Displays the account, fills, and positions reports so you can sanity-check that the downloaded continuous data is usable.

What to expect:
- `continuous_backtest_summary_df` showing the backtest scope and strategy settings.
- An account report plus fills and positions previews.
- If this cell fails because no bars were found, run the optional continuous-root download cell first.


In [ ]:
from datetime import UTC, datetime
from decimal import Decimal
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import LoggingConfig
from nautilus_trader.examples.strategies.ema_cross import EMACross
from nautilus_trader.examples.strategies.ema_cross import EMACrossConfig
from nautilus_trader.model.data import Bar
from nautilus_trader.model.enums import AccountType
from nautilus_trader.model.enums import OmsType
from nautilus_trader.model.identifiers import TraderId
from nautilus_trader.model.instruments import FuturesContract
from nautilus_trader.model.objects import Money
from nautilus_trader.serialization.arrow.serializer import ArrowSerializer

required_settings = [
    'INSTRUMENTS_PARQUET_PATH',
    'CONTINUOUS_BARS_OUTPUT_DIR',
    'ROOT_PRODUCT',
    'ROOT_EXCHANGE',
    'BACKTEST_START',
    'BACKTEST_END',
    'BACKTEST_FAST_EMA_PERIOD',
    'BACKTEST_SLOW_EMA_PERIOD',
    'BACKTEST_TRADE_SIZE',
    'BACKTEST_STARTING_BALANCE',
    'BACKTEST_REPORT_PREVIEW_ROWS',
]
missing_settings = [name for name in required_settings if name not in globals()]
if missing_settings:
    raise RuntimeError(
        'Backtest settings are missing from the active kernel. '
        'Run the notebook settings cell first. Missing: '
        + ', '.join(missing_settings)
    )

if 'format_ts_ns' not in globals():
    def format_ts_ns(value: int | None) -> str | None:
        if not value:
            return None
        return datetime.fromtimestamp(value / 1_000_000_000, tz=UTC).isoformat()

if 'normalize_utc_timestamp' not in globals():
    def normalize_utc_timestamp(value) -> pd.Timestamp:
        ts = pd.Timestamp(value)
        if ts.tzinfo is None:
            ts = ts.tz_localize('UTC')
        else:
            ts = ts.tz_convert('UTC')
        return ts

if 'read_nautilus_objects' not in globals():
    def read_nautilus_objects(path: Path, data_cls: type):
        if not path.exists():
            return []
        table = pq.read_table(path)
        objects = list(ArrowSerializer.deserialize(data_cls=data_cls, batch=table))
        if data_cls is Bar and objects and not isinstance(objects[0], Bar):
            return Bar.from_pyo3_list(objects)
        return objects

if 'is_continuous_root_instrument' not in globals():
    def is_continuous_root_instrument(instrument: FuturesContract) -> bool:
        info = getattr(instrument, 'info', {}) or {}
        return bool(info.get('is_continuous_root'))

if 'load_continuous_root_instrument' not in globals():
    def load_continuous_root_instrument(
        path: Path,
        root_symbol: str,
        exchange: str,
    ) -> FuturesContract:
        instruments = read_nautilus_objects(path, FuturesContract)
        for instrument in instruments:
            symbol = getattr(getattr(instrument, 'raw_symbol', None), 'value', None)
            instrument_exchange = getattr(instrument, 'exchange', None)
            if (
                symbol == root_symbol
                and instrument_exchange == exchange
                and is_continuous_root_instrument(instrument)
            ):
                return instrument
        raise RuntimeError(
            'Could not find the synthetic continuous root instrument in the instrument parquet. '
            'Run the discovery cell first.'
        )

if 'load_bars_from_daily_directory' not in globals():
    def load_bars_from_daily_directory(
        output_dir: Path,
        instrument_id: str | None = None,
        start=None,
        end=None,
    ) -> list[Bar]:
        if not output_dir.exists():
            return []

        start_ts = normalize_utc_timestamp(start) if start is not None else None
        end_ts = normalize_utc_timestamp(end) if end is not None else None
        bars_by_id: dict[str, Bar] = {}

        for path in sorted(output_dir.glob('*.parquet')):
            try:
                day_start = pd.Timestamp(path.stem, tz='UTC')
            except Exception:
                day_start = None

            if start_ts is not None and day_start is not None and day_start + pd.Timedelta(days=1) <= start_ts:
                continue
            if end_ts is not None and day_start is not None and day_start >= end_ts:
                continue

            for bar in read_nautilus_objects(path, Bar):
                if instrument_id is not None and str(bar.bar_type.instrument_id) != instrument_id:
                    continue
                if start_ts is not None and bar.ts_event < start_ts.value:
                    continue
                if end_ts is not None and bar.ts_event >= end_ts.value:
                    continue
                bars_by_id[f'{bar.bar_type}|{bar.ts_event}'] = bar

        return [bars_by_id[key] for key in sorted(bars_by_id)]

if BACKTEST_FAST_EMA_PERIOD >= BACKTEST_SLOW_EMA_PERIOD:
    raise ValueError('BACKTEST_FAST_EMA_PERIOD must be smaller than BACKTEST_SLOW_EMA_PERIOD.')

continuous_instrument = load_continuous_root_instrument(
    path=INSTRUMENTS_PARQUET_PATH,
    root_symbol=ROOT_PRODUCT,
    exchange=ROOT_EXCHANGE,
)
continuous_backtest_bars = load_bars_from_daily_directory(
    output_dir=CONTINUOUS_BARS_OUTPUT_DIR,
    instrument_id=str(continuous_instrument.id),
    start=BACKTEST_START,
    end=BACKTEST_END,
)

if not continuous_backtest_bars:
    raise RuntimeError(
        'No continuous root bars were found for the selected backtest window. '
        'Run the optional continuous-root download cell first.'
    )

continuous_backtest_summary_df = pd.DataFrame(
    [
        {
            'instrument_id': str(continuous_instrument.id),
            'bar_count': len(continuous_backtest_bars),
            'first_bar_utc': format_ts_ns(continuous_backtest_bars[0].ts_event),
            'last_bar_utc': format_ts_ns(continuous_backtest_bars[-1].ts_event),
            'fast_ema_period': BACKTEST_FAST_EMA_PERIOD,
            'slow_ema_period': BACKTEST_SLOW_EMA_PERIOD,
            'trade_size': BACKTEST_TRADE_SIZE,
            'starting_balance': BACKTEST_STARTING_BALANCE,
            'bar_type': str(continuous_backtest_bars[0].bar_type),
        },
    ],
)

display(continuous_backtest_summary_df)

engine = BacktestEngine(
    config=BacktestEngineConfig(
        trader_id=TraderId('BACKTESTER-001'),
        logging=LoggingConfig(log_level='ERROR', log_colors=False, use_pyo3=False),
        run_analysis=False,
    ),
)
venue = continuous_instrument.id.venue

try:
    engine.add_venue(
        venue=venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        base_currency=continuous_instrument.quote_currency,
        starting_balances=[Money(BACKTEST_STARTING_BALANCE, continuous_instrument.quote_currency)],
    )
    engine.add_instrument(continuous_instrument)
    engine.add_data(continuous_backtest_bars)
    engine.add_strategy(
        EMACross(
            config=EMACrossConfig(
                instrument_id=continuous_instrument.id,
                bar_type=continuous_backtest_bars[0].bar_type,
                trade_size=Decimal(BACKTEST_TRADE_SIZE),
                fast_ema_period=BACKTEST_FAST_EMA_PERIOD,
                slow_ema_period=BACKTEST_SLOW_EMA_PERIOD,
                subscribe_trade_ticks=False,
                subscribe_quote_ticks=False,
                request_bars=False,
            ),
        ),
    )
    engine.run()
    continuous_backtest_account_report = engine.trader.generate_account_report(venue)
    continuous_backtest_fills_report = engine.trader.generate_order_fills_report()
    continuous_backtest_positions_report = engine.trader.generate_positions_report()
finally:
    engine.dispose()

print(f'Continuous EMA backtest completed for {continuous_instrument.id}')
display(continuous_backtest_account_report)

display(
    continuous_backtest_fills_report.head(BACKTEST_REPORT_PREVIEW_ROWS)
    if hasattr(continuous_backtest_fills_report, 'head')
    else continuous_backtest_fills_report,
)
display(
    continuous_backtest_positions_report.head(BACKTEST_REPORT_PREVIEW_ROWS)
    if hasattr(continuous_backtest_positions_report, 'head')
    else continuous_backtest_positions_report,
)


,instrument_id,bar_count,first_bar_utc,last_bar_utc,fast_ema_period,slow_ema_period,trade_size,starting_balance,bar_type
0,MNQ.CME.RITHMIC,791262,2024-01-01T23:01:00+00:00,2026-03-27T21:00:00+00:00,10,20,1,250000.0,MNQ.CME.RITHMIC-1-MINUTE-LAST-EXTERNAL


## Next steps

- Change `ROOT_PRODUCT`, `ROOT_EXCHANGE`, `OUTPUT_DIR`, `BAR_START`, and `BAR_END` in the settings cell when reusing the notebook for another product family.
- Use the contract bar cell when you want instrument ids tied to real expiries such as `MNQM6.CME`.
- Use the optional continuous bar cell when you want root-symbol history tied to the synthetic backtest-only root instrument such as `MNQ.CME`.
- Use the final EMA-crossover backtest cell when you want a quick validation pass on the continuous root bars before building your own strategy.
- Replace the final `EMACross` cell with your own backtest once the continuous data path looks sane.
- Use `BAR_DAY_LIMIT` or `BAR_INSTRUMENT_LIMIT` when you want to stage downloads in smaller batches.
- Point `OUTPUT_DIR` at an existing location to merge with previously written Nautilus parquet artifacts.
- If discovery fails, inspect `discovery_summary_df` first; it is the fastest way to see which symbol candidates Rithmic rejected.
